[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/llama-certified/notebooks/day-05-prompt-formatting.ipynb#scrollTo=a1b2c3d4)

---
# Day 5 · Prompt Formatting — Chat Templates and Instruction Formatting
**certified-journeys / llama-certified** · Day 5 · Prompt Engineering

> **Goal for today:** By the end of this notebook you can manually construct a Llama 3 chat prompt using the correct special tokens, verify it against `apply_chat_template`, and implement a `PromptBuilder` class that handles both Llama 3.1 and 3.2 formats.


In [ ]:
%pip install -q transformers>=4.40.0 torch


## Step 1 · The Llama 3 Special Token Format

Llama 3 uses a strict set of special tokens to delimit turns in a conversation. Getting even one token wrong causes the model to misparse the conversation — the system prompt may be silently ignored.

| Token | Role |
|---|---|
| `<|begin_of_text|>` | Start of the entire sequence (BOS) |
| `<|start_header_id|>` | Opens a role header |
| `<|end_header_id|>` | Closes a role header |
| `<|eot_id|>` | End-of-turn — signals model to stop generating |

A complete Llama 3 prompt for a system + user + assistant turn looks like:

```
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

You are a helpful assistant.<|eot_id|><|start_header_id|>user<|end_header_id|>

Hello!<|eot_id|><|start_header_id|>assistant<|end_header_id|>

```

Note the **double newline** after each `<|end_header_id|>` — this is required. The trailing assistant header has no `<|eot_id|>`, which is the signal for the model to begin generating.


In [ ]:
# Step 1: Manually construct a Llama 3 prompt from scratch

BOS          = '<|begin_of_text|>'
HDR_OPEN     = '<|start_header_id|>'
HDR_CLOSE    = '<|end_header_id|>'
EOT          = '<|eot_id|>'

def make_turn(role: str, content: str, final: bool = False) -> str:
    """Build one conversation turn.
    
    If final=True the turn has no EOT — used for the trailing assistant header
    that prompts the model to generate its next token.
    """
    header  = f'{HDR_OPEN}{role}{HDR_CLOSE}\n\n'
    body    = content
    trailer = '' if final else EOT
    return header + body + trailer

messages = [
    {'role': 'system',    'content': 'You are a concise Python tutor.'},
    {'role': 'user',      'content': 'What is a list comprehension?'},
]

# Build the full prompt manually
parts = [BOS]
for i, msg in enumerate(messages):
    is_last = (i == len(messages) - 1)
    parts.append(make_turn(msg['role'], msg['content'], final=False))

# Append the open assistant header to trigger generation
parts.append(f'{HDR_OPEN}assistant{HDR_CLOSE}\n\n')

manual_prompt = ''.join(parts)
print('=== Manual Llama 3 Prompt ===')
print(repr(manual_prompt))


### What just happened?

- We assembled the prompt token-by-token using Python string operations.
- **Each role turn follows the exact pattern**: `HDR_OPEN + role + HDR_CLOSE + \n\n + content + EOT`.
- The final assistant header is left **open** (no EOT) — this is the generation prompt.
- A missing `\n\n` after `<|end_header_id|>` is the most common mistake and causes subtle misattribution.


## Step 2 · Using `apply_chat_template` with a Mock Tokenizer

The official way to format prompts is via `tokenizer.apply_chat_template`. The `meta-llama/Meta-Llama-3-8B-Instruct` model is gated on HuggingFace, so we use a **mock tokenizer** that reproduces the Llama 3 chat template exactly.

In production you would replace `MockLlamaTokenizer` with:
```python
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained('meta-llama/Meta-Llama-3-8B-Instruct')
```


In [ ]:
# Step 2: Mock tokenizer that implements apply_chat_template for Llama 3

class MockLlamaTokenizer:
    """Reproduces the Llama 3 chat template without requiring gated model access."""

    BOS      = '<|begin_of_text|>'
    HDR_OPEN = '<|start_header_id|>'
    HDR_CLOSE= '<|end_header_id|>'
    EOT      = '<|eot_id|>'

    def apply_chat_template(
        self,
        conversation: list[dict],
        tokenize: bool = False,
        add_generation_prompt: bool = True,
    ) -> str:
        """Returns the formatted prompt string (tokenize=False path only)."""
        result = self.BOS
        for msg in conversation:
            role    = msg['role']
            content = msg['content']
            result += f'{self.HDR_OPEN}{role}{self.HDR_CLOSE}\n\n{content}{self.EOT}'
        if add_generation_prompt:
            result += f'{self.HDR_OPEN}assistant{self.HDR_CLOSE}\n\n'
        return result


tokenizer = MockLlamaTokenizer()

template_prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)

print('=== apply_chat_template Output ===')
print(repr(template_prompt))


### What just happened?

- `apply_chat_template` returns the same token sequence we built manually.
- `tokenize=False` returns a string — pass `tokenize=True` with a real tokenizer to get input IDs directly.
- `add_generation_prompt=True` appends the open assistant header — **always use this flag at inference time**, never at training time.
- The mock lets us test template logic in CI without gated model credentials.


## Step 3 · Verify Manual == Template

A deterministic test: the manual prompt and the template output must be byte-for-byte identical. Any divergence indicates a formatting bug.


In [ ]:
# Step 3: Byte-for-byte comparison

if manual_prompt == template_prompt:
    print('PASS — manual prompt matches apply_chat_template output exactly.')
else:
    # Show the first character that differs
    for i, (a, b) in enumerate(zip(manual_prompt, template_prompt)):
        if a != b:
            print(f'FAIL — first mismatch at index {i}')
            print(f'  manual   : {repr(manual_prompt[max(0,i-10):i+20])}')
            print(f'  template : {repr(template_prompt[max(0,i-10):i+20])}')
            break
    else:
        # Same content but different lengths
        print(f'FAIL — length mismatch: manual={len(manual_prompt)}, template={len(template_prompt)}')

# Also show the human-readable formatted prompt
print('\n=== Human-readable prompt ===')
print(template_prompt)


### What just happened?

- Both approaches produce identical strings — our manual implementation is correct.
- **The printed human-readable form** reveals the double newline structure clearly.
- In a real project, this test would run in CI against the actual tokenizer to catch template drift across model versions.


## Step 4 · Raw Format vs Chat Template — Compare Outputs

What happens if you skip the chat template and feed a raw string? We simulate both and compare how a model would parse them. In production this difference causes the model to treat system instructions as user text.

| Format | System prompt respected? | Instruction following |
|---|---|---|
| Raw (no template) | No — merged with user content | Poor / unpredictable |
| Chat template | Yes — isolated in system turn | Reliable |


In [ ]:
# Step 4: Show the difference between raw format and chat template format

# Raw format — what a naive implementation might produce
raw_prompt = (
    'System: You are a concise Python tutor.\n'
    'User: What is a list comprehension?\n'
    'Assistant:'
)

# Chat template format (from Step 2)
chat_prompt = template_prompt

def analyse_prompt(label: str, prompt: str) -> None:
    print(f'=== {label} ===')
    print(f'Length         : {len(prompt)} chars')
    print(f'Has BOS token  : {"<|begin_of_text|>" in prompt}')
    print(f'Has header tags: {"<|start_header_id|>" in prompt}')
    print(f'Has EOT tokens : {"<|eot_id|>" in prompt}')
    # Count how many turns are properly delimited
    eot_count = prompt.count('<|eot_id|>')
    print(f'EOT count      : {eot_count} (expect 2 for sys+user before generation prompt)')
    print()

analyse_prompt('RAW prompt', raw_prompt)
analyse_prompt('CHAT TEMPLATE prompt', chat_prompt)

print('Conclusion:')
print('  Raw format lacks the structural tokens the model was trained to recognise.')
print('  Without them the model cannot isolate the system role from user content.')


### What just happened?

- The raw prompt is just plain text — no structural tokens for the model to parse.
- **The chat template prompt has 2 EOT tokens** (one after system, one after user), correctly separating turns.
- Instruction-tuned models are trained exclusively on the chat template format; raw prompts fall into a distribution shift that causes unreliable behavior.
- This is why using `apply_chat_template` is non-negotiable for instruction-tuned models.


## Step 5 · Multi-Turn Conversation Formatting

Real applications involve multi-turn conversations. The entire history must be reformatted on each turn — you cannot append to a previous prompt string because the EOT on the last assistant turn changes position.


In [ ]:
# Step 5: Build a multi-turn conversation and format it

conversation_history = [
    {'role': 'system',    'content': 'You are a concise Python tutor. Keep answers under 3 sentences.'},
    {'role': 'user',      'content': 'What is a list comprehension?'},
    {'role': 'assistant', 'content': 'A list comprehension is a compact way to create a list from an iterable. Example: [x*2 for x in range(5)] gives [0, 2, 4, 6, 8].'},
    {'role': 'user',      'content': 'Can I filter inside one?'},
]

multi_turn_prompt = tokenizer.apply_chat_template(
    conversation_history,
    tokenize=False,
    add_generation_prompt=True,
)

print('=== Multi-turn prompt ===')
print(multi_turn_prompt)
print(f'\nTotal turns delimited by EOT: {multi_turn_prompt.count("<|eot_id|>")}')
print('(Expect 3: system + user turn 1 + assistant turn 1)')


### What just happened?

- The full conversation history — system, user, assistant, user — is packed into a single prompt string.
- **3 EOT tokens** mark the end of each completed turn; the open assistant header signals the next generation.
- `apply_chat_template` handles the full history in one call — never manually concatenate turns from previous inference calls.
- Context window management (truncating old turns) must happen *before* calling `apply_chat_template`.


## Step 6 · Llama 3.1 vs 3.2 — Template Differences

Llama 3.2 (vision + smaller text models) introduced a `<|image|>` token for multimodal turns and adjusted how tool-call turns are formatted. The base chat structure is identical, but tool-use and vision inputs require additional handling.

| Feature | Llama 3.1 | Llama 3.2 |
|---|---|---|
| Basic chat structure | Identical | Identical |
| Tool call turns | `ipython` role | `ipython` role (same) |
| Vision inputs | Not supported | `<|image|>` token in user turn |
| System prompt | Same format | Same format |
| BOS token | `<|begin_of_text|>` | `<|begin_of_text|>` |


In [ ]:
# Step 6: PromptBuilder class that handles Llama 3.1 and 3.2 differences

from typing import Optional


class PromptBuilder:
    """Builds Llama 3.x chat prompts, handling model-version differences.
    
    Usage:
        builder = PromptBuilder(model_version='3.2')
        prompt  = builder.build(messages, image_token_count=1)
    """

    BOS       = '<|begin_of_text|>'
    HDR_OPEN  = '<|start_header_id|>'
    HDR_CLOSE = '<|end_header_id|>'
    EOT       = '<|eot_id|>'
    IMAGE_TOK = '<|image|>'   # Llama 3.2 only

    def __init__(self, model_version: str = '3.1') -> None:
        """
        Args:
            model_version: '3.1' or '3.2'
        """
        self.version = model_version
        self.supports_vision = model_version.startswith('3.2')

    def _format_turn(self, role: str, content: str) -> str:
        return f'{self.HDR_OPEN}{role}{self.HDR_CLOSE}\n\n{content}{self.EOT}'

    def build(
        self,
        messages: list[dict],
        add_generation_prompt: bool = True,
        image_token_count: int = 0,
    ) -> str:
        """Return the formatted prompt string.
        
        Args:
            messages:              List of {role, content} dicts.
            add_generation_prompt: Append open assistant header if True.
            image_token_count:     Number of <|image|> tokens to prepend to the
                                   first user message (Llama 3.2 only).
        """
        if image_token_count > 0 and not self.supports_vision:
            raise ValueError('image_token_count requires model_version="3.2"')

        result = self.BOS
        for idx, msg in enumerate(messages):
            role    = msg['role']
            content = msg['content']

            # Prepend image tokens to first user turn (Llama 3.2 vision)
            if (self.supports_vision
                    and role == 'user'
                    and image_token_count > 0
                    and idx == next((i for i, m in enumerate(messages) if m['role'] == 'user'), -1)):
                content = self.IMAGE_TOK * image_token_count + '\n' + content

            result += self._format_turn(role, content)

        if add_generation_prompt:
            result += f'{self.HDR_OPEN}assistant{self.HDR_CLOSE}\n\n'

        return result

    def validate(self, prompt: str) -> dict:
        """Run basic sanity checks on a formatted prompt."""
        return {
            'has_bos':           self.BOS in prompt,
            'has_open_header':   self.HDR_OPEN in prompt,
            'eot_count':         prompt.count(self.EOT),
            'ends_with_asst_hdr':prompt.endswith(f'{self.HDR_CLOSE}\n\n'),
            'has_double_newline':f'{self.HDR_CLOSE}\n\n' in prompt,
        }


# --- Test Llama 3.1 ---
builder_31 = PromptBuilder(model_version='3.1')
prompt_31  = builder_31.build(messages)
print('Llama 3.1 validation:', builder_31.validate(prompt_31))

# --- Test Llama 3.2 with vision ---
builder_32 = PromptBuilder(model_version='3.2')
vision_messages = [
    {'role': 'system', 'content': 'You are a vision model assistant.'},
    {'role': 'user',   'content': 'What do you see in this image?'},
]
prompt_32  = builder_32.build(vision_messages, image_token_count=1)
print('\nLlama 3.2 vision validation:', builder_32.validate(prompt_32))
print('\n3.2 prompt (vision):')
print(prompt_32)


### What just happened?

- `PromptBuilder` encapsulates all version-specific logic behind a single `build()` call.
- The `validate()` method gives a quick health check — useful in unit tests and debugging.
- **Llama 3.2 vision** prepends `<|image|>` tokens to the first user turn; the text content follows after a newline.
- **Both versions share identical base structure** — the only structural difference is the optional image token prefix.


## Step 7 · Tool Calls — The `ipython` Role

Llama 3.1+ supports function/tool calling via a special `ipython` role. Tool results are injected as an `ipython` turn, and the model then generates the final response in the `assistant` turn.


In [ ]:
# Step 7: Format a tool-call conversation (system -> user -> assistant (tool call) -> ipython -> assistant)

import json

tool_conversation = [
    {
        'role': 'system',
        'content': 'You are an assistant with access to a weather tool.'
    },
    {
        'role': 'user',
        'content': 'What is the weather in Berlin?'
    },
    {
        'role': 'assistant',
        # Model outputs a tool call as a JSON-formatted string in the assistant turn
        'content': json.dumps({
            'name': 'get_weather',
            'parameters': {'location': 'Berlin', 'unit': 'celsius'}
        })
    },
    {
        # Tool result injected as the ipython role
        'role': 'ipython',
        'content': json.dumps({'temperature': 18, 'condition': 'partly cloudy'})
    },
]

# Use PromptBuilder for the tool conversation
tool_prompt = builder_31.build(tool_conversation, add_generation_prompt=True)
print('=== Tool-call conversation prompt ===')
print(tool_prompt)
print()
print('Validation:', builder_31.validate(tool_prompt))
print(f'EOT count: {tool_prompt.count("<|eot_id|>")} (expect 4: sys+user+asst_tool+ipython)')


### What just happened?

- **Tool call flow**: user → assistant (outputs JSON tool call) → `ipython` (tool result) → assistant (final response).
- The `ipython` role uses the same header/EOT pattern as any other role — there is no special syntax.
- **4 EOT tokens** separate the 4 completed turns before the final generation prompt.
- In production, the `ipython` turn content is the raw string output from your function execution.


## Step 8 · Batch Formatting and Token Count Estimation

When serving multiple requests you need to batch-format prompts and estimate token counts before sending to the model. A rough estimate is ~3 chars per token for English text.


In [ ]:
# Step 8: Batch formatting utility

from dataclasses import dataclass


@dataclass
class PromptBatch:
    prompts: list[str]
    token_estimates: list[int]
    max_tokens: int


def batch_format(
    conversations: list[list[dict]],
    builder: PromptBuilder,
    chars_per_token: float = 3.2,
) -> PromptBatch:
    """Format a list of conversations and estimate token counts.
    
    chars_per_token: empirical estimate for Llama tokenizer (~3.2 for English).
    Production use: replace with tokenizer.encode() for exact counts.
    """
    prompts = [builder.build(conv) for conv in conversations]
    estimates = [int(len(p) / chars_per_token) for p in prompts]
    return PromptBatch(
        prompts=prompts,
        token_estimates=estimates,
        max_tokens=max(estimates) if estimates else 0,
    )


# Example: format 3 conversations
sample_conversations = [
    [{'role': 'user', 'content': 'What is 2+2?'}],
    [
        {'role': 'system', 'content': 'You are a math tutor.'},
        {'role': 'user',   'content': 'Explain the quadratic formula.'},
    ],
    [
        {'role': 'system', 'content': 'You are a code reviewer.'},
        {'role': 'user',   'content': 'Review this function: def add(a,b): return a+b'},
        {'role': 'assistant', 'content': 'The function looks correct. Consider adding type hints.'},
        {'role': 'user',   'content': 'How do I add type hints?'},
    ],
]

batch = batch_format(sample_conversations, builder_31)

for i, (prompt, est) in enumerate(zip(batch.prompts, batch.token_estimates)):
    print(f'Conv {i+1}: ~{est} tokens | {len(prompt)} chars')

print(f'\nMax tokens in batch: {batch.max_tokens}')
print('(Use this to set max_new_tokens for the batch)')


### What just happened?

- `batch_format` wraps `PromptBuilder.build()` for multiple conversations in one call.
- Token estimates use a chars-per-token heuristic (~3.2 for English with Llama's BPE tokenizer).
- **In production**, use `tokenizer.encode(prompt, return_tensors=None)` then `len()` for exact counts.
- `max_tokens` tells you the minimum context window size needed to process this entire batch.


In [ ]:
# Challenge: Implement a PromptTruncator that shortens conversation history
# to fit within a token budget, always preserving the system prompt and
# the most recent user message.
#
# Signature:
#   def truncate_to_budget(
#       messages: list[dict],
#       token_budget: int,
#       builder: PromptBuilder,
#       chars_per_token: float = 3.2,
#   ) -> list[dict]:
#
# Rules:
#   1. Always keep the system message (if any)
#   2. Always keep the last user message
#   3. Drop middle turns (oldest first) until the formatted prompt fits
#   4. Return the truncated message list (not the prompt string)
#
# Test with:
#   long_conv = [system_msg, user1, asst1, user2, asst2, user3]
#   budget = 80  # tokens
#   result = truncate_to_budget(long_conv, budget, builder_31)
#   assert result[0]['role'] == 'system'
#   assert result[-1]['role'] == 'user'

# Your solution here


---
## Day 5 key concepts recap

| Concept | What to remember |
|---|---|
| BOS token | `<|begin_of_text|>` must appear exactly once at the start |
| Header format | `HDR_OPEN + role + HDR_CLOSE + \n\n + content + EOT` |
| Generation prompt | Open assistant header (no EOT) at the end — triggers generation |
| apply_chat_template | Always use it; never concatenate turns manually |
| Llama 3.2 vision | Same base format + `<|image|>` prefix in first user turn |
| Tool calls | Use the `ipython` role to inject tool results |
| Token estimation | ~3.2 chars/token for English; use tokenizer.encode() for exact counts |

> **Tip:** Always use `apply_chat_template` rather than manually constructing the prompt. The special tokens (BOS, EOT, header_id) must be exact — a missing token causes the model to ignore the system prompt entirely.

---
## What's next
**Day 6** → Fine-Tuning with QLoRA on Consumer Hardware — adapt Llama-3.2-1B to your own dataset using 4-bit quantization and LoRA adapters.

Mark Day 5 complete in your [tracker](../index.html).
